In [ ]:
# API key -- 5361a0df0d26f9d67b2901bcd9aeb642e0908bcf

import pandas as pd
import numpy as np
import torch

In [ ]:
files = [
    'goemotions_1.csv'
]

dfs = [pd.read_csv(file) for file in files]
df_text = pd.concat(dfs, ignore_index=True)

emotion_columns = df_text.columns[df_text.columns.get_loc("admiration"):].tolist()

df_text["dominant_emotion"] = df_text[emotion_columns].idxmax(axis=1)

df_text['emotions_list'] = df_text[emotion_columns].apply(
    lambda row: [col for col, val in row.items() if val > 0],
    axis=1
)

df_text['emotions_list'] = df_text.apply(
    lambda row: [row['dominant_emotion']] if len(row['emotions_list']) == 0 else row['emotions_list'],
    axis=1
)

df_oryginal = df_text[['text', 'dominant_emotion', 'emotions_list']]

In [ ]:
df = df_oryginal[:10000]

In [ ]:
top10 = df['dominant_emotion'].value_counts().nlargest(10).index
allowed = set(top10)

df_class = df[df['dominant_emotion'].isin(top10)].copy()

df_class['emotions_list'] = df_class['emotions_list'].apply(
    lambda labels: [l for l in labels if l in allowed]
)

print(df_class['dominant_emotion'].value_counts())

In [ ]:
!pip install scikit-multilearn==0.2.0

In [ ]:
from sklearn.utils import resample

groups = df_class.groupby('dominant_emotion')
max_count = groups.size().max()

oversampled = [
    resample(g,
             replace=True,
             n_samples=max_count,
             random_state=42)
    for _, g in groups
]

df_balanced = pd.concat(oversampled).reset_index(drop=True)
df_balanced = df_balanced[df_balanced['dominant_emotion'].isin(top10)].copy()
df_balanced['emotions_list'] = df_balanced['emotions_list'].apply(
    lambda labels: [l for l in labels if l in allowed]
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
df_class['dominant_emotion'].value_counts().plot(
    kind='bar', color='steelblue'
)
plt.title('Rozkład klas przed oversamplingiem')
plt.xlabel('Emocja')
plt.ylabel('Liczność')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('')

plt.figure(figsize=(8, 5))
df_balanced['dominant_emotion'].value_counts().plot(
    kind='bar', color='steelblue'
)
plt.title('Rozkład klas po oversamplingu')
plt.xlabel('Emocja')
plt.ylabel('Liczność')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset
from sklearn.preprocessing import MultiLabelBinarizer
from skmultilearn.model_selection import iterative_train_test_split
from transformers import BertTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

mlb = MultiLabelBinarizer()
X_arr_train = df_balanced['text'].to_numpy().reshape(-1, 1)
Y_arr_train = mlb.fit_transform(df_balanced['emotions_list'])

X_train, y_train, X_val, y_val = iterative_train_test_split(
    X_arr_train, Y_arr_train, test_size=0.1
)

train_texts = X_train.flatten()
val_texts   = X_val.flatten()
train_labels = y_train
val_labels   = y_val

X_arr_test = df_class['text'].to_numpy().reshape(-1, 1)
Y_arr_test = mlb.transform(df_class['emotions_list'])

_, _, X_test, y_test = iterative_train_test_split(
    X_arr_test, Y_arr_test, test_size=0.2
)

test_texts  = X_test.flatten()
test_labels = y_test

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

class EmotionDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(
            list(texts), truncation=True, padding=True, max_length=256
        )
        self.labels = torch.tensor(labels, dtype=torch.float)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

train_dataset = EmotionDataset(train_texts, train_labels)
val_dataset   = EmotionDataset(val_texts, val_labels)
test_dataset  = EmotionDataset(test_texts, test_labels)

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)
from transformers import BertForSequenceClassification, Trainer, TrainingArguments

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=len(mlb.classes_),
    problem_type="multi_label_classification"
).to(device)

training_args = TrainingArguments(
    output_dir='./output',
    save_strategy='epoch',
    eval_strategy='epoch',
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    logging_dir='./logs',
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

results = trainer.evaluate()
print(results)

In [ ]:
from sklearn.metrics import f1_score, hamming_loss, accuracy_score, classification_report

val_predictions = trainer.predict(val_dataset)
val_probs = torch.sigmoid(torch.tensor(val_predictions.predictions)).numpy()
val_labels = val_predictions.label_ids.astype(int)

best_thr, best_f1 = 0.0, 0.0
for thr in np.arange(0.05, 0.55, 0.05):
    preds = (val_probs > thr).astype(int)
    f1 = f1_score(val_labels, preds, average="micro", zero_division=0)
    if f1 > best_f1:
        best_f1, best_thr = f1, thr

test_predictions = trainer.predict(test_dataset)
test_probs = torch.sigmoid(torch.tensor(test_predictions.predictions)).numpy()
test_labels = test_predictions.label_ids.astype(int)

test_preds = (test_probs > best_thr).astype(int)

print("F1 micro:", f1_score(test_labels, test_preds, average="micro", zero_division=0))
print("F1 macro:", f1_score(test_labels, test_preds, average="macro", zero_division=0))
print("Hamming loss:", hamming_loss(test_labels, test_preds))
print("Subset accuracy:", accuracy_score(test_labels, test_preds))

from sklearn.preprocessing import MultiLabelBinarizer
print("\nClassification report per label:")
print(classification_report(test_labels, test_preds, target_names=mlb.classes_, zero_division=0))

In [ ]:
def emotions_to_ids(emotions):
    return [list(mlb.classes_).index(e) for e in emotions]

df_labels = df_class.copy()
df_labels['label'] = df_labels['emotions_list'].apply(emotions_to_ids)
df_labels

In [ ]:
labels = list(mlb.classes_)
labels_dict = {i: e for i, e in enumerate(labels)}
decode = lambda vec: [labels_dict[i] for i,v in enumerate(vec) if v==1]

y_true = np.array(test_labels)
conf   = (probs * y_pred).sum(axis=1) / np.maximum(y_pred.sum(axis=1), 1)

def dominant(vec):
    vec = np.array(vec)
    return None if vec.sum()==0 else labels_dict[int(np.argmax(vec))]

results_df = pd.DataFrame({
    'text': test_texts.tolist(),
    'y_true_vec': list(y_true),
    'y_pred_vec': list(y_pred),
    'conf': conf
})
results_df['actual_emotions']  = results_df['y_true_vec'].apply(decode)
results_df['predict_emotions'] = results_df['y_pred_vec'].apply(decode)
results_df['dominant_predict_emotion'] = results_df['y_pred_vec'].apply(dominant)

finally_df = results_df[['text','actual_emotions','predict_emotions',
                         'dominant_predict_emotion','y_true_vec','y_pred_vec','conf']
                        ].sample(frac=1, random_state=42).reset_index(drop=True)

finally_df

In [ ]:
SEED = 78
np.random.seed(SEED)

empty_mask = finally_df['predict_emotions'].str.len() == 0
empty_idx = np.random.choice(finally_df.index[empty_mask],
                             size=min(2, empty_mask.sum()), replace=False)
empty_preds = finally_df.loc[empty_idx].copy()

nonempty = finally_df['predict_emotions'].str.len() > 0
low_pool = finally_df[nonempty].nsmallest(10, 'conf').index
k = min(3, len(low_pool))
low_idx = np.random.choice(low_pool, size=k, replace=False)
borderline_df = finally_df.loc[low_idx].copy()

def same_labels_row(r):
    yt = np.array(r['y_true_vec']); yp = np.array(r['y_pred_vec'])
    return set(np.where(yt==1)[0]) == set(np.where(yp==1)[0])

correct_idx = finally_df.index[finally_df.apply(same_labels_row, axis=1)]
k = min(5, len(correct_idx))
sel_correct_idx = np.random.choice(correct_idx, size=k, replace=False)
correct_df = finally_df.loc[sel_correct_idx].copy()

selected_df = pd.concat([empty_preds, borderline_df, correct_df]).reset_index(drop=True)
selected_df

In [ ]:
!pip install shap

In [ ]:
num_labels = model.config.num_labels

from itertools import chain
class_names = sorted({e for lst in df_labels['emotions_list'] for e in lst})

id2label = {i: name for i, name in enumerate(class_names)}
label2id = {name: i for i, name in id2label.items()}

model.config.num_labels = len(class_names)
model.config.id2label = {i: name for i, name in enumerate(class_names)}
model.config.label2id = {name: i for i, name in model.config.id2label.items()}

def predict_proba(texts):
    enc = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.sigmoid(logits).cpu().numpy()
    return probs

In [ ]:
def predict_proba2(texts):
    if isinstance(texts, str):
        texts = [texts]
    elif isinstance(texts, np.ndarray):
        texts = texts.tolist()
    elif not isinstance(texts, list):
        raise ValueError("texts must be str or list[str]")

    texts = [str(t) for t in texts if isinstance(t, str) and t.strip()]

    if not texts:
        raise ValueError("No valid text inputs")

    enc = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.sigmoid(logits).cpu().numpy()
    return probs

In [ ]:
import shap
import torch
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

threshold = 0.30

masker = shap.maskers.Text(tokenizer)
explainer_shap = shap.Explainer(predict_proba2, masker, output_names=class_names)

sample_texts = selected_df['text'].dropna().astype(str).tolist()
pred_probs = predict_proba2(sample_texts)

for i, row in selected_df.iterrows():
    text = str(row['text'])

    pred_labels = row['predict_emotions']
    true_labels = row['actual_emotions']
    dominant = row['dominant_predict_emotion']

    if not pred_labels:
        print(f"\nSample #{i} — brak przewidzianych etykiet (pomijam wykres)")
        continue

    top_ids = [class_names.index(e) for e in pred_labels]
    shap_values = explainer_shap([text])
    shap_values_filtered = shap_values[:, :, top_ids]

    print(f"\nSample #{i}")
    print(f"  Actual: {true_labels}")
    print(f"  Pred: {pred_labels}")
    print(f"  Dominant pred class: {dominant}")

    shap.plots.text(shap_values_filtered)
    print("\n\n\n\n")

In [ ]:
!pip install lime

In [ ]:
from IPython.core.display import display, HTML

custom_css = """
<style>
/* probability wykres */
.lime.predict_proba text {
    fill: white !important;
}

/* explanation wykres */
.lime.explanation text {
    fill: white !important;
}

/* tekst w sekcji "Text with highlighted words" */
.lime.text_div {
    color: white !important;
}
.lime.text_div span {
    color: white !important;
}
</style>
"""

In [ ]:
from IPython.display import display, HTML
import numpy as np
import os
from lime.lime_text import LimeTextExplainer
import torch

class_names = [id2label[i] for i in range(len(id2label))]
explainer_lime = LimeTextExplainer(class_names=class_names, random_state=0)

os.makedirs("lime_text_reports", exist_ok=True)

threshold = 0.30

for i, row in selected_df.iterrows():
    text_instance = row["text"]

    predicted_ids = [class_names.index(e) for e in row["predict_emotions"]]

    if not predicted_ids:
        print(f"\nSample #{i} — brak przewidzianych etykiet (pomijam wykres)")
        continue

    exp = explainer_lime.explain_instance(
        text_instance,
        predict_proba,
        num_features=10,
        num_samples=5000,
        labels=predicted_ids
    )

    print(f"\n#{i}")
    print(f"  Actual: {row['actual_emotions']}")
    print(f"  Pred: {row['predict_emotions']}")
    print(f"  Dominant pred class: {row['dominant_predict_emotion']}")
    print('')
    exp_html = exp.as_html()
    display(HTML(custom_css + exp_html))
    print('')

In [ ]:
max_size_finally_df = finally_df['text']

In [ ]:
from collections import defaultdict
from tqdm import tqdm
from collections import Counter


def shap_word_ranking(text, cls):
    sv = explainer_shap([text]).values[0, :, cls]
    toks = tokenizer.tokenize(text.lower())
    sv = sv[:len(toks)]
    words, contribs = [], []
    buf, vals = '', []
    for tok, val in zip(toks, sv):
        if tok.startswith('##'):
            buf += tok[2:]
            vals.append(val)
        else:
            if buf:
                words.append(buf)
                contribs.append(np.mean(vals))
            buf, vals = tok, [val]
    if buf:
        words.append(buf)
        contribs.append(np.mean(vals))

    contribs = np.array(contribs)
    order = np.argsort(-contribs)

    return words, list(order)


def to_positions_from_types(ranked_types, words):
    from collections import defaultdict
    pos = defaultdict(list)
    for i,w in enumerate(words): pos[w].append(i)
    seen = defaultdict(int)
    idxs = []
    for w in ranked_types:
        k = seen[w]
        if k < len(pos[w]): idxs.append(pos[w][k]); seen[w]+=1
    left = [i for i in range(len(words)) if i not in set(idxs)]
    return idxs + left

def lime_word_ranking(text, cls):
    exp = explainer_lime.explain_instance(text, predict_proba, labels=[cls])
    feats = exp.as_list(label=cls)
    feats_sorted = sorted(feats, key=lambda x: x[1], reverse=True)
    words = [w for w, _ in feats_sorted]
    scores = dict(feats_sorted)
    return words, scores


def align_ranking_to_words(ranking, words):
    return [w for w in ranking if w in words]


def get_words(text):
    toks = tokenizer.tokenize(text.lower())
    words = []
    buf = ''
    for t in toks:
        if t.startswith('##'):
            buf += t[2:]
        else:
            if buf:
                words.append(buf)
            buf = t
    if buf:
        words.append(buf)
    return words


def deletion_curve(text, rank_idx, cls, steps=10):
    import math
    words = get_words(text); n = len(words)
    step = math.ceil(n/steps); probs = []
    for k in range(0, steps+1):
        keep = set(rank_idx[k*step:])
        masked = [words[i] for i in range(n) if i in keep]
        probs.append(predict_proba([" ".join(masked)])[0][cls])
    probs = np.array(probs)
    return probs/probs[0] if probs[0]>0 else probs


def insertion_curve(text, rank_idx, cls, steps=10):
    import math
    words = get_words(text); n = len(words)
    step = math.ceil(n/steps); probs = []
    for k in range(0, steps+1):
        take = set(rank_idx[:k*step])
        cur = [words[i] for i in range(n) if i in take]
        probs.append(predict_proba([" ".join(cur)])[0][cls])
    p_end = predict_proba([" ".join(words)])[0][cls]
    probs = np.array(probs)
    return probs/p_end if p_end>0 else probs


def bootstrap_band(curves, n_boot=1000, ci=95):
    X = np.vstack(curves)
    N, T = X.shape
    lo, hi = np.empty(T), np.empty(T)
    for t in range(T):
        means = [X[np.random.randint(0, N, N), t].mean() for _ in range(n_boot)]
        lo[t], hi[t] = np.percentile(means, [(100-ci)/2, 100-(100-ci)/2])
    return X.mean(0), lo, hi


def random_ranking(text):
    w = get_words(text)
    np.random.shuffle(w)
    return w

def freq_ranking(text):
    w = get_words(text)
    return sorted(w, key=lambda x: corpus_counts.get(x, 0), reverse=True)

def tfidf_ranking(text):
    from sklearn.feature_extraction.text import TfidfVectorizer
    tfidf = TfidfVectorizer(tokenizer=str.split, preprocessor=None, lowercase=False)
    tfidf.fit(corpus_words)
    w = " ".join(get_words(text))
    scores = tfidf.transform([w]).toarray()[0]
    vocab = np.array(tfidf.get_feature_names_out())
    return [vocab[i] for i in np.argsort(scores)[::-1] if scores[i] > 0]


corpus_tok = [get_words(t) for t in max_size_finally_df]

corpus_counts = Counter(w for doc in corpus_tok for w in doc)

corpus_words = [" ".join(get_words(t)) for t in max_size_finally_df]

methods = {
    'SHAP': shap_word_ranking,
    'LIME': lime_word_ranking,
    'Random': lambda text, cls: (random_ranking(text), {}),
    'Freq':   lambda text, cls: (freq_ranking(text), {}),
    'TFIDF':  lambda text, cls: (tfidf_ranking(text), {})
}

max_size_finally_df = finally_df['text']
steps = 30
texts = max_size_finally_df

In [ ]:
import re
from collections import defaultdict
import numpy as np
from tqdm import tqdm
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def merge_subtokens(tokens, values):
    """Scal subtokens typu ##ville w pełne słowa i uśrednij ich wartości SHAP"""
    words, contribs = [], []
    buf, vals = "", []

    for tok, val in zip(tokens, values):
        if tok.startswith("##"):
            buf += tok[2:]
            vals.append(val)
        else:
            if buf:
                words.append(buf)
                contribs.append(np.mean(vals))
            buf, vals = tok, [val]

    if buf:
        words.append(buf)
        contribs.append(np.mean(vals))

    return words, contribs


def clean_word(word):
    """Usuń artefakty, cyfry i znaki specjalne"""
    word = re.sub(r"[^a-zA-Z]", "", word)
    word = word.lower()
    if len(word) < 2:
        return ""
    return word


def global_shap_ranking_for_classes_clean(texts, labels, class_names, selected_classes, topk=10, batch_size=32, sample_size=200):
    results = {}
    n = min(len(texts), sample_size)

    total_batches = len(range(0, n, batch_size)) * len(selected_classes)

    with tqdm(total=total_batches, desc="Całkowity postęp") as pbar:
        for cls_name in selected_classes:
            cls = class_names.index(cls_name)
            contribs = defaultdict(list)

            for i in range(0, n, batch_size):
                batch_texts = texts[i:i+batch_size]
                batch_labels = labels[i:i+batch_size]

                shap_values = explainer_shap(batch_texts)

                for j, text in enumerate(batch_texts):
                    if batch_labels[j][cls] != 1:
                        continue

                    tokens = tokenizer.tokenize(text.lower())
                    sv = shap_values.values[j][:len(tokens), cls]

                    words, vals = merge_subtokens(tokens, sv)

                    for w, v in zip(words, vals):
                        w_clean = clean_word(w)
                        if not w_clean:
                            continue
                        if w_clean in stop_words:
                            continue
                        contribs[w_clean].append(v)

                pbar.update(1)

            avg_contribs = {w: np.mean(vals) for w, vals in contribs.items()}
            ranked = sorted(avg_contribs.items(), key=lambda x: x[1], reverse=True)
            results[cls_name] = ranked[:topk]

    return results


selected_classes = ["anger", "gratitude", "admiration"]
df_global = global_shap_ranking_for_classes_clean(
    test_texts,
    test_labels,
    class_names,
    selected_classes,
    topk=10,
    batch_size=32,
    sample_size=500
)

for cls, ranking in df_global.items():
    print(f"\nTop słowa dla klasy '{cls}':")
    for w, score in ranking:
        print(f"  {w}: {score:.4f}")

In [ ]:
def deletion_auc(text, ranking, cls, steps=20):
    words = get_words(text)
    ranking = [w for w in ranking if w in words]
    n = len(words)
    if n == 0:
        return 0.0
    step = max(1, n // steps)

    base_prob = predict_proba([" ".join(words)])[0][cls]
    if base_prob <= 0:
        return 0.0

    probs = [base_prob]
    for k in range(1, steps+1):
        remove = set(ranking[:k*step])
        masked = [w for w in words if w not in remove]
        p = predict_proba([" ".join(masked)])[0][cls]
        probs.append(p)

    probs = np.array(probs) / base_prob
    probs = np.clip(probs, 0, 1)

    xs = np.linspace(0, 1, len(probs))
    auc = np.trapz(probs, xs)
    return auc

def insertion_auc(text, ranking, cls, steps=20):
    words = get_words(text)
    ranking = [w for w in ranking if w in words]
    n = len(words)
    if n == 0:
        return 0.0
    step = max(1, n // steps)

    full_prob = predict_proba([" ".join(words)])[0][cls]
    if full_prob <= 0:
        return 0.0

    probs = [predict_proba([""])[0][cls]]
    for k in range(1, steps+1):
        add = set(ranking[:k*step])
        cur = [w for w in words if w in add]
        p = predict_proba([" ".join(cur)])[0][cls]
        probs.append(p)

    probs = np.array(probs) / full_prob
    probs = np.clip(probs, 0, 1)

    xs = np.linspace(0, 1, len(probs))
    auc = np.trapz(probs, xs)
    return auc


def get_positive_classes(y_row):
    return np.where(y_row == 1)[0]


records_ml = []
records_top1 = []

for text, y_row in tqdm(list(zip(max_size_finally_df, y_true)), desc="Liczenie AUC"):
    probs = predict_proba([text])[0]

    for cls in get_positive_classes(y_row):
        for name, ranker in methods.items():
            ranked_words, _ = ranker(text, cls)
            auc_del = deletion_auc(text, ranked_words, cls)
            auc_ins = insertion_auc(text, ranked_words, cls)
            records_ml.append({'method': name, 'cls': cls,
                               'auc_del': auc_del, 'auc_ins': auc_ins})

    top_cls = int(np.argmax(probs))
    for name, ranker in methods.items():
        ranked_words, _ = ranker(text, top_cls)
        auc_del = deletion_auc(text, ranked_words, top_cls)
        auc_ins = insertion_auc(text, ranked_words, top_cls)
        records_top1.append({'method': name, 'cls': top_cls,
                             'auc_del': auc_del, 'auc_ins': auc_ins})

results_ml = pd.DataFrame(records_ml)
results_top1 = pd.DataFrame(records_top1)

In [ ]:
import scipy.stats as st

def stats_summary(results_df, label):
    print(f"\n=== {label} ===")

    shap_del = results_df.loc[results_df.method=='SHAP','auc_del'].sort_index()
    lime_del = results_df.loc[results_df.method=='LIME','auc_del'].sort_index()
    shap_ins = results_df.loc[results_df.method=='SHAP','auc_ins'].sort_index()
    lime_ins = results_df.loc[results_df.method=='LIME','auc_ins'].sort_index()

    summary = (results_df.groupby('method')
                          .agg({'auc_del':['mean','std'],
                                'auc_ins':['mean','std']}))
    print(summary)

stats_summary(results_ml, "Multi-label (tylko klasy pozytywne)")
stats_summary(results_top1, "Top-1 (najpewniejsza klasa)")

In [ ]:
def compare_vs_baseline(results, baseline='Random', metric='auc_del', better='lower', seed=42):
    rng = np.random.default_rng(seed)
    out = {}
    for m in ['SHAP','LIME']:
        vals_m = results.loc[results.method==m, metric].values
        vals_b_all = results.loc[results.method==baseline, metric].values

        if len(vals_b_all) >= len(vals_m):
            vals_b = vals_b_all[:len(vals_m)]
        else:
            vals_b = rng.choice(vals_b_all, size=len(vals_m), replace=True)

        if better == 'lower':
            frac = np.mean(vals_m < vals_b)
        else:
            frac = np.mean(vals_m > vals_b)
        out[m] = frac
    return out


print("\n--- Porównanie XAI vs Random ---")
print("AUC_del:", compare_vs_baseline(results_top1, 'Random', 'auc_del', better='lower'))
print("AUC_ins:", compare_vs_baseline(results_top1, 'Random', 'auc_ins', better='higher'))

print("\n--- Porównanie XAI vs TFIDF ---")
print("AUC_del:", compare_vs_baseline(results_top1, 'TFIDF', 'auc_del', better='lower'))
print("AUC_ins:", compare_vs_baseline(results_top1, 'TFIDF', 'auc_ins', better='higher'))

In [ ]:
!pip install cliffs-delta

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from cliffs_delta import cliffs_delta

def bootstrap_ci(data, n_boot=1000, ci=95):
    means = [np.mean(np.random.choice(data, size=len(data), replace=True))
             for _ in range(n_boot)]
    lo, hi = np.percentile(means, [(100-ci)/2, 100-(100-ci)/2])
    return np.mean(data), lo, hi

def analyze_results(results, label):
    print(f"\n==============================")
    print(f"Analiza statystyczna: {label}")
    print(f"==============================")

    for metric in ['auc_del','auc_ins']:
        print(f"\n--- Bootstrap 95% CI dla {metric} ---")
        for method in results['method'].unique():
            vals = results.loc[results['method']==method, metric].values
            m, lo, hi = bootstrap_ci(vals)
            print(f"{method:7s} mean={m:.3f}  95%CI=({lo:.3f},{hi:.3f})")

    def paired_stats(a, b, label):
        stat, p = wilcoxon(a, b)
        d = (a.mean()-b.mean()) / np.sqrt(((a.std()**2)+(b.std()**2))/2)
        delta, _ = cliffs_delta(a.tolist(), b.tolist())
        print(f"{label}: Wilcoxon p={p:.4f}, Cohen's d={d:.3f}, Cliff's delta={delta:.3f}")

    for metric in ['auc_del','auc_ins']:
        shap_vals = results.loc[results['method']=='SHAP', metric].sort_index().values
        lime_vals = results.loc[results['method']=='LIME', metric].sort_index().values
        rand_vals = results.loc[results['method']=='Random', metric].sort_index().values

        print(f"\n--- Testy dla {metric} ---")
        paired_stats(shap_vals, lime_vals, "SHAP vs LIME")
        paired_stats(
            np.concatenate([shap_vals,lime_vals]),
            np.concatenate([rand_vals[:len(shap_vals)],rand_vals[:len(lime_vals)]]),
            "XAI vs Random"
        )

analyze_results(results_ml, "Multi-label (tylko klasy pozytywne)")
analyze_results(results_top1, "Top-1 (najpewniejsza klasa)")

In [ ]:
curves_del_ml = defaultdict(list)
curves_ins_ml = defaultdict(list)
curves_del_top1 = defaultdict(list)
curves_ins_top1 = defaultdict(list)

In [ ]:
for text in tqdm(texts, desc="Krzywe Top-1"):
    cls = int(np.argmax(predict_proba([text])[0]))
    for name in ['SHAP','LIME','Random','Freq','TFIDF']:

        if name == 'SHAP':
            words_list, rank_idx = shap_word_ranking(text, cls)
        else:
            words_list = get_words(text)
            if name == 'LIME':
                types, _ = lime_word_ranking(text, cls)
            elif name == 'Random':
                types = random_ranking(text)
            elif name == 'Freq':
                types = freq_ranking(text)
            else:
                types = tfidf_ranking(text)
            types = [w for w in types if w in words_list]
            rank_idx = to_positions_from_types(types, words_list)

        curves_del_top1[name].append(deletion_curve(text, rank_idx, cls, steps))


def plot_curves(curves_top1, title_prefix, steps=10):
    xs = np.linspace(0, 100, steps+1)
    fig, ax = plt.subplots(1, 1, figsize=(8,6), dpi=100)
    for m in ['SHAP','LIME','Random','Freq','TFIDF']:
        mean, lo, hi = bootstrap_band(curves_top1[m], n_boot=1000)
        ax.plot(xs, mean, label=m)
        ax.fill_between(xs, lo, hi, alpha=0.2)
    ax.set_xlabel('% słów')
    ax.set_ylabel('średnia $\~p$ / p0' if title_prefix=='Deletion' else 'średnia $\~p$ / p_end')
    ax.set_title(f'{title_prefix} — Top-1')
    ax.legend()
    plt.tight_layout()
    plt.show()

plot_curves(curves_del_top1, "Deletion", steps)

In [ ]:
for text in tqdm(texts, desc="Krzywe Top-1"):
    cls = int(np.argmax(predict_proba([text])[0]))
    for name in ['SHAP','LIME','Random','Freq','TFIDF']:

        if name == 'SHAP':
            words_list, rank_idx = shap_word_ranking(text, cls)
        else:
            words_list = get_words(text)
            if name == 'LIME':
                types, _ = lime_word_ranking(text, cls)
            elif name == 'Random':
                types = random_ranking(text)
            elif name == 'Freq':
                types = freq_ranking(text)
            else:
                types = tfidf_ranking(text)
            types = [w for w in types if w in words_list]
            rank_idx = to_positions_from_types(types, words_list)

        curves_ins_top1[name].append(insertion_curve(text, rank_idx, cls, steps))

def plot_curves(curves_top1, title_prefix, steps=10):
    xs = np.linspace(0, 100, steps+1)
    fig, ax = plt.subplots(1, 1, figsize=(8,6), dpi=100)
    for m in ['SHAP','LIME','Random','Freq','TFIDF']:
        mean, lo, hi = bootstrap_band(curves_top1[m], n_boot=1000)
        ax.plot(xs, mean, label=m)
        ax.fill_between(xs, lo, hi, alpha=0.2)
    ax.set_xlabel('% słów')
    ax.set_ylabel('średnia $\~p$ / p0' if title_prefix=='Deletion' else 'średnia $\~p$ / p_end')
    ax.set_title(f'{title_prefix} — Top-1')
    ax.legend()
    plt.tight_layout()
    plt.show()

plot_curves(curves_ins_top1, "Insertion", steps)

In [ ]:
for text, y_row in tqdm(list(zip(texts, y_true)), desc="Krzywe Multi-label"):
    probs = predict_proba([text])[0]

    for cls in np.where(y_row == 1)[0]:
        if probs[cls] < 0.35:
            continue

        for name in ['SHAP','LIME','Random','Freq','TFIDF']:

            if name == 'SHAP':
                words_list, rank_idx = shap_word_ranking(text, cls)
            else:
                words_list = get_words(text)
                if name == 'LIME':
                    types, _ = lime_word_ranking(text, cls)
                elif name == 'Random':
                    types = random_ranking(text)
                elif name == 'Freq':
                    types = freq_ranking(text)
                else:
                    types = tfidf_ranking(text)

                types = [w for w in types if w in words_list]
                rank_idx = to_positions_from_types(types, words_list)

            curves_del_ml[name].append(deletion_curve(text, rank_idx, cls, steps))

def plot_curves(curves_ml, title_prefix, steps=10):
    xs = np.linspace(0, 100, steps+1)
    fig, ax = plt.subplots(1, 1, figsize=(8,6), dpi=100)
    for m in ['SHAP','LIME','Random','Freq','TFIDF']:
        if len(curves_ml[m]) == 0:
            continue
        mean, lo, hi = bootstrap_band(curves_ml[m], n_boot=1000)
        ax.plot(xs, mean, label=m)
        ax.fill_between(xs, lo, hi, alpha=0.2)
    ax.set_xlabel('% słów')
    ax.set_ylabel('średnia $\~p$ / p0' if title_prefix=='Deletion' else 'średnia $\~p$ / p_end')
    ax.set_title(f'{title_prefix} — Multi-label')
    ax.legend()
    plt.tight_layout()
    plt.show()

plot_curves(curves_del_ml, "Deletion", steps)

In [ ]:
for text, y_row in tqdm(list(zip(texts, y_true)), desc="Krzywe Multi-label"):
    probs = predict_proba([text])[0]

    for cls in np.where(y_row == 1)[0]:
        p0 = probs[cls]
        if p0 < 0.35:
            continue

        for name in ['SHAP','LIME','Random','Freq','TFIDF']:
            words_list = get_words(text)

            if name == 'SHAP':
                words_list, rank_idx = shap_word_ranking(text, cls)
            else:
                if name == 'LIME':
                    types, _ = lime_word_ranking(text, cls)
                elif name == 'Random':
                    types = random_ranking(text)
                elif name == 'Freq':
                    types = freq_ranking(text)
                else:
                    types = tfidf_ranking(text)
                types = [w for w in types if w in words_list]
                rank_idx = to_positions_from_types(types, words_list)

            curves_ins_ml[name].append(insertion_curve(text, rank_idx, cls, steps))

def plot_curves(curves_ml, title_prefix, steps=10):
    xs = np.linspace(0, 100, steps+1)
    fig, ax = plt.subplots(1, 1, figsize=(8,6), dpi=100)
    for m in ['SHAP','LIME','Random','Freq','TFIDF']:
        if len(curves_ml[m]) == 0:
            continue
        mean, lo, hi = bootstrap_band(curves_ml[m], n_boot=1000)
        ax.plot(xs, mean, label=m)
        ax.fill_between(xs, lo, hi, alpha=0.2)
    ax.set_xlabel('% słów')
    ax.set_ylabel('średnia $\~p$ / p0' if title_prefix=='Deletion' else 'średnia $\~p$ / p_end')
    ax.set_title(f'{title_prefix} — Multi-label')
    ax.legend()
    plt.tight_layout()
    plt.show()

plot_curves(curves_ins_ml, "Insertion", steps)